# Meteorite Identification - EDA

这个 notebook 用于对 `meteorite-identification` 数据集做初步探索，目标包括：

1. 给出数据集基本描述  
2. 检查标签分布与文件对应关系  
3. 评估是否需要清理数据  
4. 随机可视化若干正负样本  
5. 为后续 baseline 建模提供依据  

默认数据路径：
`/mnt/sharedata/ssd_large/users/liyx/kaggle/meteorite-identification`


In [ ]:
import os
from pathlib import Path
import random
import warnings

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
SEED = 42
random.seed(SEED)
np.random.seed(SEED)


In [ ]:
DATA_ROOT = Path("/mnt/sharedata/ssd_large/users/liyx/kaggle/meteorite-identification")
TRAIN_IMG_DIR = DATA_ROOT / "train_images"
TEST_IMG_DIR = DATA_ROOT / "test_images"
TRAIN_CSV = DATA_ROOT / "train_labels.csv"
SAMPLE_SUB = DATA_ROOT / "sample_submission.csv"

for p in [DATA_ROOT, TRAIN_IMG_DIR, TEST_IMG_DIR, TRAIN_CSV, SAMPLE_SUB]:
    print(f"{p}: {'OK' if p.exists() else 'MISSING'}")


## 1. 读取训练标签与 submission 模板

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
sample_sub_df = pd.read_csv(SAMPLE_SUB)

print("train_df shape:", train_df.shape)
print("sample_sub_df shape:", sample_sub_df.shape)
display(train_df.head())
display(sample_sub_df.head())


## 2. 数据集基本描述

这里先确认：
- 训练集 / 测试集大小
- 标签列是否存在
- 是否有重复 id
- 文件与 csv 是否一一对应


In [ ]:
train_files = sorted([p.name for p in TRAIN_IMG_DIR.iterdir() if p.is_file()])
test_files = sorted([p.name for p in TEST_IMG_DIR.iterdir() if p.is_file()])

summary = {
    "num_train_rows_csv": len(train_df),
    "num_train_images_dir": len(train_files),
    "num_test_rows_template": len(sample_sub_df),
    "num_test_images_dir": len(test_files),
    "train_duplicate_ids": int(train_df["id"].duplicated().sum()),
    "test_duplicate_ids_in_template": int(sample_sub_df["id"].duplicated().sum()),
}
pd.Series(summary, name="value")


In [ ]:
train_csv_ids = set(train_df["id"].tolist())
train_img_ids = set(train_files)
test_template_ids = set(sample_sub_df["id"].tolist())
test_img_ids = set(test_files)

train_missing_image = sorted(train_csv_ids - train_img_ids)
train_unlisted_image = sorted(train_img_ids - train_csv_ids)

test_missing_image = sorted(test_template_ids - test_img_ids)
test_unlisted_image = sorted(test_img_ids - test_template_ids)

check_df = pd.DataFrame({
    "split": ["train", "train", "test", "test"],
    "issue": ["csv_has_no_image", "image_not_in_csv", "template_has_no_image", "image_not_in_template"],
    "count": [len(train_missing_image), len(train_unlisted_image), len(test_missing_image), len(test_unlisted_image)]
})
display(check_df)

if len(train_missing_image) > 0:
    print("Examples train_missing_image:", train_missing_image[:5])
if len(train_unlisted_image) > 0:
    print("Examples train_unlisted_image:", train_unlisted_image[:5])
if len(test_missing_image) > 0:
    print("Examples test_missing_image:", test_missing_image[:5])
if len(test_unlisted_image) > 0:
    print("Examples test_unlisted_image:", test_unlisted_image[:5])


## 3. 标签分布

训练集标签是否平衡会影响：
- loss 设计
- metric 选择
- 是否需要 class weights / focal loss


In [ ]:
label_counts = train_df["label"].value_counts().sort_index()
label_ratio = train_df["label"].value_counts(normalize=True).sort_index()

display(pd.DataFrame({
    "count": label_counts,
    "ratio": label_ratio.round(4)
}))

ax = label_counts.plot(kind="bar", rot=0, title="Train Label Distribution")
ax.set_xlabel("label")
ax.set_ylabel("count")
plt.show()


## 4. 图像基础统计

抽样检查：
- 图像尺寸
- 图像 mode（RGB / 非 RGB）
- 是否存在损坏文件
- 是否存在异常长宽比


In [ ]:
def inspect_images(file_list, image_dir, sample_size=None, desc="Inspecting"):
    records = []
    broken = []

    if sample_size is not None:
        file_list = file_list[:sample_size]

    for fname in tqdm(file_list, desc=desc):
        path = image_dir / fname
        try:
            with Image.open(path) as img:
                img.load()
                width, height = img.size
                mode = img.mode
                records.append({
                    "id": fname,
                    "width": width,
                    "height": height,
                    "mode": mode,
                    "aspect_ratio": round(width / height, 4) if height > 0 else np.nan,
                })
        except Exception as e:
            broken.append({"id": fname, "error": str(e)})
    return pd.DataFrame(records), pd.DataFrame(broken)

train_stats_df, train_broken_df = inspect_images(train_files, TRAIN_IMG_DIR, desc="Train images")
test_stats_df, test_broken_df = inspect_images(test_files, TEST_IMG_DIR, desc="Test images")

print("train_stats_df:", train_stats_df.shape)
print("test_stats_df:", test_stats_df.shape)
print("broken train images:", len(train_broken_df))
print("broken test images:", len(test_broken_df))


In [ ]:
def summarize_image_stats(stats_df, name):
    summary = pd.Series({
        "num_images": len(stats_df),
        "width_mean": round(stats_df["width"].mean(), 2),
        "height_mean": round(stats_df["height"].mean(), 2),
        "width_min": int(stats_df["width"].min()),
        "width_max": int(stats_df["width"].max()),
        "height_min": int(stats_df["height"].min()),
        "height_max": int(stats_df["height"].max()),
        "num_unique_sizes": int(stats_df[["width", "height"]].drop_duplicates().shape[0]),
        "num_unique_modes": int(stats_df["mode"].nunique()),
    }, name=name)
    return summary

display(pd.concat([
    summarize_image_stats(train_stats_df, "train"),
    summarize_image_stats(test_stats_df, "test")
], axis=1))


In [ ]:
print("Train image modes:")
display(train_stats_df["mode"].value_counts())

print("Test image modes:")
display(test_stats_df["mode"].value_counts())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(train_stats_df["width"], bins=30)
axes[0].set_title("Train Width Distribution")
axes[0].set_xlabel("width")

axes[1].hist(train_stats_df["height"], bins=30)
axes[1].set_title("Train Height Distribution")
axes[1].set_xlabel("height")

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(train_stats_df["width"], train_stats_df["height"], alpha=0.4, s=12)
plt.title("Train Image Size Distribution")
plt.xlabel("width")
plt.ylabel("height")
plt.show()


## 5. 是否需要清理数据？

这里给出几个与清理相关的检查：
- 文件缺失 / 多余
- 损坏图片
- 重复 id
- 标签异常
- 极端尺寸样本


In [ ]:
issues = {
    "train_duplicate_ids": int(train_df["id"].duplicated().sum()),
    "test_duplicate_ids_in_template": int(sample_sub_df["id"].duplicated().sum()),
    "train_missing_image_count": len(train_missing_image),
    "train_unlisted_image_count": len(train_unlisted_image),
    "test_missing_image_count": len(test_missing_image),
    "test_unlisted_image_count": len(test_unlisted_image),
    "broken_train_count": len(train_broken_df),
    "broken_test_count": len(test_broken_df),
    "invalid_train_labels": int((~train_df["label"].isin([0, 1])).sum()),
}
pd.Series(issues, name="count")


In [ ]:
extreme_small = train_stats_df[(train_stats_df["width"] < 64) | (train_stats_df["height"] < 64)]
extreme_ratio = train_stats_df[(train_stats_df["aspect_ratio"] > 2.5) | (train_stats_df["aspect_ratio"] < 0.4)]

print("Very small train images:", len(extreme_small))
print("Extreme aspect-ratio train images:", len(extreme_ratio))

display(extreme_small.head())
display(extreme_ratio.head())


### 初步清理建议

一般只有在以下情况出现时，才建议显式清理：
- 存在损坏图片
- csv 与图像目录不匹配
- 存在异常标签
- 有极少数尺寸离谱、明显错误的样本

否则，通常**不需要手动删数据**，而是通过：
- `convert("RGB")`
- 统一 resize
- 常规 augmentation

来处理。


## 6. 随机可视化若干正负样本

这里分别随机展示 `label=0` 和 `label=1` 的若干训练样本，帮助判断：
- 两类是否有明显形态差异
- 是否存在背景 shortcut
- 是否有标注异常 / 可疑样本


In [ ]:
def show_random_samples(df, image_dir, label, n=8, seed=42):
    sub_df = df[df["label"] == label].sample(n=min(n, (df["label"] == label).sum()), random_state=seed)
    paths = sub_df["id"].tolist()

    cols = 4
    rows = int(np.ceil(len(paths) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.array(axes).reshape(-1)

    for ax, fname in zip(axes, paths):
        img = Image.open(image_dir / fname).convert("RGB")
        ax.imshow(img)
        ax.set_title(f"{fname}\nlabel={label}")
        ax.axis("off")

    for ax in axes[len(paths):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

show_random_samples(train_df, TRAIN_IMG_DIR, label=0, n=8, seed=SEED)
show_random_samples(train_df, TRAIN_IMG_DIR, label=1, n=8, seed=SEED)


## 7. 可选：按类别看尺寸差异

如果两类图像在分辨率、拍摄方式等方面存在系统差异，模型可能会学到 shortcut。


In [ ]:
train_stats_with_label = train_stats_df.merge(train_df, on="id", how="left")

display(
    train_stats_with_label.groupby("label")[["width", "height", "aspect_ratio"]]
    .agg(["mean", "std", "min", "max"])
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for label in [0, 1]:
    subset = train_stats_with_label[train_stats_with_label["label"] == label]
    axes[0].hist(subset["width"], bins=25, alpha=0.5, label=f"label={label}")
    axes[1].hist(subset["height"], bins=25, alpha=0.5, label=f"label={label}")

axes[0].set_title("Width by Label")
axes[1].set_title("Height by Label")
axes[0].legend()
axes[1].legend()
plt.tight_layout()
plt.show()


## 8. EDA 小结模板

运行完以上单元后，可以结合结果填写这份总结：

- 训练集大小：  
- 测试集大小：  
- 标签是否平衡：  
- 图片尺寸是否统一：  
- 是否全为 RGB：  
- 是否存在损坏图片：  
- 是否需要显式数据清理：  
- baseline 模型建议：  
- 推荐主指标：  

通常若数据整体干净，第一版 baseline 可采用：
- `ResNet18 / ResNet50`
- 输入尺寸 `224x224`
- `CrossEntropyLoss`
- `AdamW`
- 主监控指标 `val_f1`
